### Calculation of EarthQuake's Impact Damage Potential

### Load The Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/ImpactSense_Oct25/data/preprocessed_earthquake_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109848 entries, 0 to 109847
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   latitude   109848 non-null  float64
 1   longitude  109848 non-null  float64
 2   depth      109848 non-null  float64
 3   mag        109848 non-null  float64
 4   magType    109848 non-null  object 
 5   rms        109848 non-null  float64
 6   type       109848 non-null  object 
 7   status     109848 non-null  object 
 8   Year       109848 non-null  int64  
 9   Month      109848 non-null  int64  
 10  Day        109848 non-null  int64  
 11  Hour       109848 non-null  int64  
 12  Minute     109848 non-null  int64  
 13  Second     109848 non-null  int64  
 14  DayOfWeek  109848 non-null  int64  
dtypes: float64(5), int64(7), object(3)
memory usage: 12.6+ MB


### 1. Magnitude Type Prioritization and Standardization

In [3]:
def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5 # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    # Add more conversions as needed
    elif mag_type.lower() == 'ml':
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type
        return np.nan

In [4]:
# Apply the conversion to the DataFrame
import numpy as np
df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
df[['mag', 'magType', 'Mw']].sample(10)

,mag,magType,Mw
80565,-0.921328,mb,2.582711
82028,-0.503699,mb,2.862522
104094,0.540372,mww,0.540372
45502,0.122744,mw,0.122744
76360,-0.921328,mb,2.582711
43448,-0.712513,mb,2.722616
2322,1.814139,mw,1.814139
81585,-0.712513,mb,2.722616
71907,-0.712513,mwc,-0.712513
24052,-0.712513,mb,2.722616


In [5]:
df[['mag', 'magType', 'Mw']].sample(10)

,mag,magType,Mw
69751,-0.921328,mwc,-0.921328
58302,1.793257,mwc,1.793257
25949,-0.086071,mb,3.142333
6942,1.020645,mw,1.020645
71001,1.375629,mwc,1.375629
14054,2.628514,mw,2.628514
39621,0.958000,mw,0.958000
83557,-0.921328,mb,2.582711
50237,-0.712513,ms,-0.948139
32830,-0.294885,mw,-0.294885


#### 2. Calculate Damage Potential Using HAZUS-Style Formula

In [6]:
def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8 # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    # Return the final damage potential score
    return damage_potential

In [7]:
# Apply the conversion to the DataFrame
df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
df[['Mw', 'depth', 'damage_potential']].head(10)

,Mw,depth,damage_potential
0,3.296720,-0.430517,10.000000
1,2.920854,-0.430517,10.000000
2,4.716657,-0.291162,10.000000
3,4.299028,-0.430517,10.000000
4,3.442890,-0.430517,10.000000
5,3.860518,-0.476968,10.000000
6,2.670277,-0.476968,10.000000
7,2.190004,-0.430517,9.574581
8,2.712040,-0.384065,10.000000
9,2.231767,-0.430517,9.678841


#### Observations:
* Exponential Energy Release: Since energy release multiplies by roughly 31.6 for every single-unit increase in magnitude, even a slight rise in $M_w$ indicates a massive increase in destructive power.Shutterstock
* Impact of Depth: Shallow earthquakes (less than 30 km deep) are significantly more destructive because seismic waves have less distance to travel and weaken before striking the surface.
* Reliability of Scales: Traditional scales like body-wave ($m_b$), surface-wave ($M_s$), and local magnitude ($M_l$) tend to "saturate" and underestimate the largest events. Only Moment Magnitude ($M_w$) remains accurate for earthquakes of all sizes.
* Calculating Damage: By combining magnitude and depth, we estimate Peak Ground Acceleration (PGA) to create a normalized 0–10 damage potential score, ensuring the assessment is both realistic and quantifiable.